# MindBridge — UlizaLlama3 GPU Model Server (Colab / Kaggle)
This notebook serves the fine-tuned **UlizaLlama3 + MindBridge LoRA adapter** over an authenticated private HTTPS endpoint.

### Instructions:
1. Go to **Runtime -> Change runtime type** and select **T4 GPU**.
2. Run the cells below in order.
3. Copy the generated `trycloudflare.com` URL into your `.env` file as `CONVERSATION_URL`.

In [ ]:
# Step 1: Install inference and tunneling dependencies
!pip install -q "transformers==4.57.6" "peft==0.21.2" bitsandbytes accelerate fastapi uvicorn pydantic python-dotenv
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
print("Dependencies installed.")

In [ ]:
# Step 2: Set credentials and environment variables
import os
os.environ["MINDBRIDGE_ALLOW_SUPERVISED"] = "1"
os.environ["MODEL_SERVICE_TOKEN"] = "mindbridge-secure-local-model-token-32chars-min"
# If your HF adapter is private, insert your HF token below:
# os.environ["HF_TOKEN"] = "hf_..."
print("Environment configured.")

In [ ]:
# Step 3: Launch FastAPI server in background
import subprocess, time
server_proc = subprocess.Popen(["python", "-m", "uvicorn", "model_server.app:app", "--host", "127.0.0.1", "--port", "8001"])
time.sleep(5)
print("FastAPI server running on port 8001.")

In [ ]:
# Step 4: Expose server via Cloudflare Tunnel
import re
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("="*60)
    print("SUCCESS! LIVE GPU ENDPOINT IS READY:")
    print(f"Set in your .env:")
    print(f"CONVERSATION_ENABLED=1")
    print(f"CONVERSATION_URL={public_url}/v1/reply")
    print(f"CONVERSATION_API_TOKEN=mindbridge-secure-local-model-token-32chars-min")
    print("="*60)
else:
    print("Could not extract tunnel URL. Check tunnel logs.")